# 04 — Fitting II: gradient descent

*Notebook 4 of 6 — Logistic Regression*

NB 3 built the **bowl** — the log-loss, a convex valley whose bottom is the best weights. This notebook
*reaches* that bottom, automatically, by rolling downhill. It is the course's **first optimizer**, and
the very engine that trains neural networks later on — so we build it by hand and watch it work.

**Prerequisites.** NB 3: the log-loss bowl $L(w)$ — convex, one bottom. NB 1: the sigmoid, and the score
as log-odds. NB 2: standardization, and why scale matters.

**What you'll be able to do.**
- Read the **gradient** as the slope of the loss, and step **against** it.
- Apply the update rule $w \leftarrow w - \eta\,\nabla L$ with a **learning rate** $\eta$.
- Use the log-loss gradient $(P - y)\,x$ — and check it is right.
- Run gradient descent by hand until the weights reach the bottom of the bowl.
- Explain what the learning rate does: crawl, converge, or diverge.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

from ml_course import colors, datasets, viz

viz.use_course_style()
np.random.seed(0)


def sigmoid(z):
    """Map any real score z to a probability in (0, 1)."""
    return 1.0 / (1.0 + np.exp(-z))


# One feature, standardized by hand (z-score) — the same setup as NB 3.
df = datasets.load_penguins()
bill = df["bill_length_mm"].to_numpy(dtype=float)
x = (bill - bill.mean()) / bill.std()
y = (df["species"] == "Gentoo").to_numpy().astype(float)


def log_loss(w, b):
    """Mean log-loss over the data for weight w and bias b."""
    p = sigmoid(w * x + b)
    eps = 1e-12
    return -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))


def grad(w, b):
    """Gradient of the mean log-loss: (dL/dw, dL/db) = (mean[(P-y)x], mean[P-y])."""
    p = sigmoid(w * x + b)
    return np.mean((p - y) * x), np.mean(p - y)


print(f"{len(y)} penguins | Gentoo (y=1): {int(y.sum())}")

## Where we are

NB 3 handed us a **bowl**: a convex loss $L$ whose lowest point is the best set of weights. In NB 1 and
NB 2 we *chose* weights by hand; now we want to **reach** the bottom on our own.

The trick is the oldest one for getting downhill in fog: feel which way the ground slopes under your
feet, take a step the other way, and repeat. Done with the loss instead of a hillside, that is
**gradient descent** — the course's first optimizer, and the same method that trains the neural networks
of chapters 11–12.

## The gradient is the slope

On a 1-D bowl, the derivative $L'(w)$ is the **slope** of the loss at the current weight. If the slope is
**positive**, the loss climbs to the right, so we should go **left**; if it is **negative**, we go
**right**. Either way, we step **against** the slope.

How far? A small fraction of it, set by the **learning rate** $\eta$ (a small positive number):

$$ w \leftarrow w - \eta\, L'(w). $$

Big slope, big step; small slope, small step. Let us see one step.

In [ ]:
w_grid = np.linspace(-2, 14, 400)
L_curve = np.array([log_loss(w, 0.0) for w in w_grid])  # 1-param bowl, bias held at 0 (as in NB 3)

w0 = 1.0
L0 = log_loss(w0, 0.0)
slope = grad(w0, 0.0)[0]          # dL/dw at w0 — the slope of the bowl there
eta_demo = 5.0
w1 = w0 - eta_demo * slope         # one downhill step (slope < 0 here, so we move right)

fig, ax = plt.subplots(figsize=(7, 4.6))
ax.plot(w_grid, L_curve, color=colors.COLORS["model"], linewidth=2)
ax.scatter([w0], [L0], color=colors.COLORS["highlight"], zorder=6, s=60)
tx = np.array([w0 - 2.5, w0 + 2.5])
ax.plot(
    tx,
    L0 + slope * (tx - w0),
    color=colors.COLORS["text"],
    linewidth=1.3,
    linestyle="--",
    label=f"slope L'(w) = {slope:.2f}",
)
ax.annotate(
    "",
    xy=(w1, log_loss(w1, 0.0)),
    xytext=(w0, L0),
    arrowprops=dict(arrowstyle="-|>", color=colors.COLORS["highlight"], linewidth=2.5),
)
ax.set_ylim(0, 1.6)
ax.set_xlabel("weight w   (bias held at b = 0)")
ax.set_ylabel("log-loss  L(w)")
ax.set_title("The gradient is the slope; step the opposite way")
ax.legend(loc="upper right")
plt.show()

**Read the figure.** The dashed line is the **tangent** — its slope is the gradient at the current
weight. Here the slope is negative (the bowl still falls to the right), so stepping *against* it moves us
**right**, toward the bottom (the rose arrow). Notice the self-regulation built in: far from the bottom
the bowl is steep, so steps are large; near the bottom it flattens, so steps shrink and the descent
**slows as it arrives**.

## The gradient of the log-loss is $(P - y)\,x$

With two weights $(w, b)$ the gradient is a pair of slopes — one per knob:

$$ \frac{\partial L}{\partial w} = \operatorname{mean}\big[(P - y)\,x\big], \qquad
   \frac{\partial L}{\partial b} = \operatorname{mean}\big[P - y\big]. $$

Read it as **error times input**: $(P - y)$ is how wrong the predicted probability is, and $x$ is the
feature that produced it. The sigmoid's derivative **cancels** in the algebra (NB 3's teaser come true),
which is what leaves this strikingly clean form — and clean gradients make for stable descent.

We will **state** this gradient and **check** it numerically; the full line-by-line derivation is in the
references.

In [ ]:
w_t, b_t = 2.0, -0.3
gw, gb = grad(w_t, b_t)
h = 1e-6
fd_w = (log_loss(w_t + h, b_t) - log_loss(w_t - h, b_t)) / (2 * h)
fd_b = (log_loss(w_t, b_t + h) - log_loss(w_t, b_t - h)) / (2 * h)
print(f"at (w, b) = ({w_t}, {b_t}):")
print(f"  dL/dw : (P-y)x = {gw:+.6f}   finite-diff = {fd_w:+.6f}   err {abs(gw - fd_w):.1e}")
print(f"  dL/db : (P-y)  = {gb:+.6f}   finite-diff = {fd_b:+.6f}   err {abs(gb - fd_b):.1e}")

## The algorithm

Gradient descent is three lines, repeated:

1. compute the gradient $\nabla L = (\partial L/\partial w,\ \partial L/\partial b)$ at the current
   weights;
2. step **against** it: $(w, b) \leftarrow (w, b) - \eta\,\nabla L$;
3. repeat.

Over the two-dimensional loss **surface** $L(w, b)$ this traces a **path** that walks downhill to the
single bottom — and NB 3's convexity promises there is only one bottom to find.

In [ ]:
def gradient_descent(eta, n_iter, w=0.0, b=0.0):
    """Full-batch gradient descent from (w, b); returns the path and the loss at each step."""
    path, losses = [(w, b)], [log_loss(w, b)]
    for _ in range(n_iter):
        gw, gb = grad(w, b)
        w -= eta * gw
        b -= eta * gb
        path.append((w, b))
        losses.append(log_loss(w, b))
    return np.array(path), np.array(losses)


path, losses = gradient_descent(eta=2.0, n_iter=400)

ww = np.linspace(-1, 10, 200)
bb = np.linspace(-4, 3, 200)
WW, BB = np.meshgrid(ww, bb)
LL = np.array([[log_loss(w, b) for w in ww] for b in bb])

fig, ax = plt.subplots(figsize=(7, 5.2))
cf = ax.contourf(WW, BB, LL, levels=25, cmap=colors.CMAP_PROBA)
fig.colorbar(cf, ax=ax, label="log-loss  L(w, b)")
ax.plot(
    path[:, 0], path[:, 1], color=colors.COLORS["highlight"], linewidth=1.5, label="descent path"
)
ax.scatter(path[::20, 0], path[::20, 1], color=colors.COLORS["highlight"], s=14, zorder=5)
ax.scatter(
    [path[0, 0]], [path[0, 1]], color=colors.COLORS["text"], s=70, zorder=6, label="start (0, 0)"
)
ax.scatter(
    [path[-1, 0]],
    [path[-1, 1]],
    color=colors.COLORS["highlight"],
    marker="*",
    s=180,
    edgecolor=colors.COLORS["text"],
    zorder=7,
    label="minimum",
)
ax.set_xlabel("weight w")
ax.set_ylabel("bias b")
ax.set_title("Gradient descent rolls downhill to the bottom")
ax.legend(loc="upper right")
plt.show()

**Read the figure.** Bluer means higher loss; the pale hollow is the bottom of the bowl. From the start
at $(0, 0)$ the path steps **across the contours, straight downhill**, curving into the hollow and
settling at the minimum (the star). Each dot is one update. The steps are long where the surface is
steep and shrink as the gradient fades near the bottom — the descent eases itself to a stop.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.4))
ax.plot(np.arange(len(losses)), losses, color=colors.COLORS["model"], linewidth=2)
ax.axhline(
    losses[-1],
    color=colors.COLORS["highlight"],
    linewidth=1.2,
    linestyle="--",
    label=f"floor ≈ {losses[-1]:.3f}",
)
ax.set_xlabel("iteration")
ax.set_ylabel("log-loss  L")
ax.set_title("The loss falls to a floor: convergence")
ax.legend()
plt.show()

**Read the figure.** The loss drops **steeply at first** — far from the bottom the slope is big, so each
step buys a lot — then **levels off at a floor**, the bottom of the bowl. That flattening is what we mean
by **convergence**: once we are at the bottom the gradient is near zero, steps become tiny, and more
iterations stop helping.

In [ ]:
# Run to convergence, then compare with sklearn's solver.
path_conv, _ = gradient_descent(eta=2.0, n_iter=5000)
w_gd, b_gd = path_conv[-1]

# Parity is against the UNREGULARIZED fit (C=inf); default C=1 is regularized (NB 5's knob).
clf = LogisticRegression(C=np.inf, max_iter=200000, tol=1e-10).fit(x.reshape(-1, 1), y)
w_sk, b_sk = clf.coef_[0, 0], clf.intercept_[0]

pd.DataFrame(
    {"weight w": [w_gd, w_sk], "bias b": [b_gd, b_sk]},
    index=["by-hand gradient descent", "LogisticRegression(C=inf)"],
).round(5)

**Read the result.** By-hand gradient descent lands on the **same weights** the library's solver finds.
The library is not magic: it walks the same convex bowl, with cleverer step rules for speed. We have now
**fitted** a logistic-regression model end to end — defined the loss (NB 3) and minimized it (here) — by
hand.

## The learning rate is the dial

One number controls the whole descent — the learning rate $\eta$:

- too **small**: the steps are tiny, the descent **crawls** and may not arrive in time;
- **well-chosen**: it **glides** smoothly to the floor;
- too **big**: a step overshoots the bottom — the loss jumps *up* instead of settling, and the
  descent never reaches the floor (on a steeper loss it would run off to infinity).

Let us watch all three.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.6))
runs = [
    (0.1, "eta = 0.1  (crawls)", ":", colors.COLORS["muted"]),
    (2.0, "eta = 2  (glides)", "-", colors.COLORS["model"]),
    (400.0, "eta = 400  (overshoots)", "--", colors.COLORS["error"]),
]
for eta, label, style, col in runs:
    _, losses_eta = gradient_descent(eta, 60)
    ax.plot(
        np.arange(len(losses_eta)), losses_eta, linewidth=2, linestyle=style, color=col, label=label
    )
ax.set_ylim(0.1, 1.3)
ax.set_xlabel("iteration")
ax.set_ylabel("log-loss  L")
ax.set_title("The learning rate: too small crawls, well-chosen glides, too big overshoots")
ax.legend()
plt.show()

**Read the figure.** The small rate (dotted) inches down and is still far from the floor when the good
rate (solid) has long since arrived. The good rate glides straight to the bottom. The large rate (dashed)
**overshoots** — its very first step sends the loss jumping *up* the wall, and it then bounces high
above the floor instead of settling, with spikes that run off the top of the chart. (The usable range is wide here because this loss
is gently curved; on **raw**, un-standardized `bill_length` it collapses to a knife-edge — a rate of
$0.003$ barely moves and $0.005$ already sends the loss climbing the wrong way — another reason NB 2
standardized.)

## Your turn

1. **Turn the dial.** In `gradient_descent`, try $\eta = 0.3$ and $\eta = 5$. Re-plot the loss-vs-iteration
   curve. Which is faster to the floor? Which is bouncier? Describe what you see.
2. **One step by hand.** Start at $w = 0$, $b = 0$. For a single penguin with $x = 1.5$ that is truly
   Gentoo ($y = 1$), the model gives $P = \sigma(0) = 0.5$. Compute the one-example gradient $(P - y)\,x$
   and the updated $w$ after one step with $\eta = 0.5$. Which direction did $w$ move, and why?
3. **A bad start.** Run the descent from $w = -5$, $b = 0$. Does it still reach the same bottom? Predict
   first (recall the bowl is **convex**), then check.

## What you built

- The **gradient** — the direction the loss rises fastest; for the log-loss it is $(P - y)\,x$ (error
  times input), which we **verified** numerically.
- The **update rule** $w \leftarrow w - \eta\,\nabla L$, and **gradient descent**: step against the
  gradient, repeat, roll to the convex bottom.
- The **learning rate** $\eta$ as the dial between crawling, gliding, and diverging.
- **Parity** with `LogisticRegression` — a model fitted, end to end, by hand.

**Vocabulary.** *gradient* · *learning rate $\eta$* · *gradient descent* · *iteration / step* ·
*convergence* · *overshoot*.

You can now train a logistic-regression model from nothing but a loss and a slope. Next we hand the job
to the real estimator and explore the knobs it gives us.

## Going further (optional)

We used the **whole dataset** to compute each gradient — *batch* gradient descent. On large data we would
instead estimate the gradient from a small random **mini-batch** each step (*stochastic* gradient
descent): the same idea, with noisier but far cheaper steps. And this is exactly how **neural networks**
learn — each unit is a logistic-style neuron, and the gradients are found layer by layer with the chain
rule (**backpropagation**, chapters 11–12). The bowl you rolled down is the same one, in many more
dimensions.

A look ahead: **NB 5** meets the real `LogisticRegression` estimator — its parameters $C$ and `l1_ratio`,
the multinomial/softmax extension to more than two classes, and how to tune them honestly.

## References

- Cox, D. R. (1958). *The regression analysis of binary sequences.* Journal of the Royal Statistical
  Society B, 20(2), 215–242. DOI: 10.1111/j.2517-6161.1958.tb00292.x
- Bishop, C. M. (2006). *Pattern Recognition and Machine Learning*, §4.3.3 (logistic regression, its
  gradient and iterative fitting). Springer.
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* (§4.4).
  DOI: 10.1007/978-0-387-84858-7

---

*Previous: 03 — Fitting I: what we optimize (log-loss)*  ·  *Next: 05 — The estimator & its parameters*